# Step 5 - BL maps and X-line

Maps of dVz = Vz - Vz_MSH in the external BL (`uniform_depthonly_spectro > 0.83`), projected on the magnetopause, for a given IMF clock angle (gaussian weights, std 30°) and dipole tilt (std 10°): weighted KNN (`knn_weighted_feature_map`). The X-line is the dVz = 0 line where the flow diverges (`find_stagnation_line`, sigma=7).

Input: output of step 4. Outputs: `{OUT}/xlines/DeltaVz_cla{cla}_tilt{tilt}.pkl` (169 files) and `{OUT}/xlines/DeltaVz_cla{cla}.pkl` (13 files), used by step 6, and the paper figures in `{OUT}/figures/`.

`prepare_data`, `get_weights_gaussian` and `knn_weighted_feature_map` were defined in FinalMapsPaper cell 2 with the same code as in swapp; they are imported from `blxline.maps`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # makes the blxline package importable
from blxline.paths import OUT, MODEL_LINES  # all outputs of this repo are written under OUT (never over the original files)

In [ ]:
# [from Maps/FinalMapsPaper.ipynb, cell 0]
import pandas as pd
import numpy as np
from blxline.maps import make_mp_grid, gaussian_filter_nan_datas, plot_CLA_sector
from blxline.maps import prepare_data, get_weights_gaussian, knn_weighted_feature_map
import matplotlib.pyplot as plt
from spok.models.planetary import Magnetosheath
from spok.plot import planet_env
from spok.plot.planet_env import make_theta_and_phi, _find_theta_for_x_slice
from spok.models.planetary import mp_shue1997
from blxline.maps import reposition, make_slice
from spok.models.planetary import get_tilt
from datetime import timedelta
from scipy.stats import binned_statistic_2d
from blxline.maps import find_stagnation_line


In [ ]:
# [from Maps/FinalMapsPaper.ipynb, cell 632]
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree


def label_pieces(y, z, max_gap=2.):
    """Label of the piece each point belongs to (-1 for NaN points)."""
    y, z = np.asarray(y, dtype=float), np.asarray(z, dtype=float)
    labels = np.full(y.shape, -1)
    valid = np.isfinite(y) & np.isfinite(z)
    pts = np.array([y[valid], z[valid]]).T
    if len(pts) == 0:
        return labels
    pairs = cKDTree(pts).query_pairs(max_gap, output_type='ndarray')
    graph = coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(len(pts), len(pts)))
    labels[valid] = connected_components(graph, directed=False)[1]
    return labels


def piece_extent(y, z):
    """Size of a piece: diagonal of its bounding box."""
    return np.hypot(np.ptp(y), np.ptp(z))


def keep_main_part(y, z, max_gap=2., min_fraction=None, return_mask=False):
    """Remove the pieces of the line (y, z) that are far from its main part.
    y, z : positions of the points of the line (the order is kept)
    max_gap : points closer than max_gap are in the same piece. Should be larger than the
        gaps you accept in a line, and smaller than the distance of the pieces to remove.
    min_fraction : if given, also keep the pieces whose extent is at least min_fraction times
        the extent of the main piece.
    Returns the cleaned y, z (and the boolean mask of the kept points if return_mask).
    """
    y, z = np.asarray(y, dtype=float), np.asarray(z, dtype=float)
    labels = label_pieces(y, z, max_gap=max_gap)
    pieces = np.unique(labels[labels >= 0])
    if len(pieces) == 0:
        mask = np.zeros(y.shape, dtype=bool)
    else:
        extents = np.array([piece_extent(y[labels == p], z[labels == p]) for p in pieces])
        if min_fraction is None:
            kept = pieces[[np.argmax(extents)]]
        else:
            kept = pieces[extents >= min_fraction * extents.max()]
        mask = np.isin(labels, kept)
    if return_mask:
        return y[mask], z[mask], mask
    return y[mask], z[mask]

## X-lines on the (clock angle, tilt) grid, used by step 6

In [ ]:
# [from Maps/FinalMapsPaper.ipynb, cell 628]
from blxline.maps import gaussian_filter_nan_1d
df = prepare_data(pd.read_pickle(f'{OUT}/BL_data_for_maps_without_cusps_withoutNp<3_all_sat_1min_2025-05-06.pkl'))
df = df[df.uniform_depthonly_spectro.values>0.83]
Xmp, Ymp, Zmp = make_mp_grid(N_grid=100,coord='cartesian')

std_cla = 30/180*np.pi
std_tilt = 10*np.pi/180

for cla in [120,125,130,135,140,145,150,155,160,165,170,175,180]:
    for tilt in [-30,-25,-20,-15,-10,-5, 0,5,10,15,20,25,30]:
        df2 = get_weights_gaussian(df, [('omni_CLA',cla*np.pi/180,std_cla),('tilt',tilt*np.pi/180,std_tilt)])
        df2['weights']=df2.weights.values/np.nansum(df2.weights.values)
        temp = df2[df2.weights.values>0]
        mapped_feature, valid = knn_weighted_feature_map(temp,'dVz','weights',
                                                 np.array([Xmp.flatten(), Ymp.flatten(), Zmp.flatten()]).T, 
                                                 k_neighbors=1000,max_distance_barycenter=1.5,median_distance=3)
        mapped_feature[valid==0]=np.nan
        mapped_feature[Xmp.flatten()<0.01] = np.nan

        y_measure,z_measure = find_stagnation_line(mapped_feature.reshape(Xmp.shape),Y_mp=Ymp, Z_mp=Zmp, sigma=7)
        z_measure = gaussian_filter_nan_1d(z_measure,1)

        results = {'CLA':cla,'std_CLA':std_cla,'tilt':tilt,'std_tilt':std_tilt,'DeltaVz':mapped_feature.reshape(Xmp.shape),'Yxline':y_measure,'Zxline':z_measure}
        pd.to_pickle(results, f'{OUT}/xlines/DeltaVz_cla{cla}_tilt{tilt}.pkl')
        print(f'Done for CLA={cla}° and tilt={tilt}°')


In [ ]:
# [from Maps/FinalMapsPaper.ipynb, cell 629]
from blxline.maps import gaussian_filter_nan_1d
df = prepare_data(pd.read_pickle(f'{OUT}/BL_data_for_maps_without_cusps_withoutNp<3_all_sat_1min_2025-05-06.pkl'))
df = df[df.uniform_depthonly_spectro.values>0.83]
Xmp, Ymp, Zmp = make_mp_grid(N_grid=100,coord='cartesian')

std_cla = 30/180*np.pi

for cla in [120,125,130,135,140,145,150,155,160,165,170,175,180]:
    df2 = get_weights_gaussian(df, [('omni_CLA',cla*np.pi/180,std_cla)])
    df2['weights']=df2.weights.values/np.nansum(df2.weights.values)
    temp = df2[df2.weights.values>0]
    mapped_feature, valid = knn_weighted_feature_map(temp,'dVz','weights',
                                             np.array([Xmp.flatten(), Ymp.flatten(), Zmp.flatten()]).T, 
                                             k_neighbors=2000,max_distance_barycenter=1.5,median_distance=3)
    mapped_feature[valid==0]=np.nan
    mapped_feature[Xmp.flatten()<0.01] = np.nan

    y_measure,z_measure = find_stagnation_line(mapped_feature.reshape(Xmp.shape),Y_mp=Ymp, Z_mp=Zmp, sigma=7)
    z_measure = gaussian_filter_nan_1d(z_measure,1)

    results = {'CLA':cla,'std_CLA':std_cla,'DeltaVz':mapped_feature.reshape(Xmp.shape),'Yxline':y_measure,'Zxline':z_measure}
    pd.to_pickle(results, f'{OUT}/xlines/DeltaVz_cla{cla}.pkl')
    print(f'Done for CLA={cla}°')


## Paper figures: maps for several clock angles and tilts

In [ ]:
# [from Maps/FinalMapsPaper.ipynb, cell 303]
df = prepare_data(pd.read_pickle(f'{OUT}/BL_data_for_maps_without_cusps_withoutNp<3_all_sat_1min_2025-05-06.pkl'))
df = df[df.uniform_depthonly_spectro.values>0.83]
Xmp, Ymp, Zmp = make_mp_grid(N_grid=100,coord='cartesian')

std_cla = np.pi/6
clas = np.linspace(np.pi/2+std_cla,np.pi,7)
min_clas = (clas-np.pi/6)[::-1]
max_clas = (clas+np.pi/6)[::-1]
nb_sectors = len(min_clas)

# Example data
fig, ax = plt.subplots(ncols=2)

ys, zs = [],[]
median_clas = []
for i, (min_cla, max_cla) in enumerate(zip(min_clas,max_clas)):
    std_cla = (max_cla-min_cla)/2
    center_cla = (max_cla+min_cla)/2

    df2 = get_weights_gaussian(df, [('omni_CLA',center_cla,std_cla)])
    df2['weights']=df2.weights.values/np.nansum(df2.weights.values)
    temp = df2[df2.weights.values>0]
    mapped_feature, valid = knn_weighted_feature_map(temp,'dVz','weights',np.array([Xmp.flatten(), Ymp.flatten(), Zmp.flatten()]).T, k_neighbors=2000,max_distance_barycenter=1.5,median_distance=3)
    mapped_feature[valid==0]=np.nan
    mapped_feature[Xmp.flatten()<0.01] = np.nan
    mapped_feature = gaussian_filter_nan_datas((mapped_feature).reshape(Xmp.shape),7)
    y, z = find_stagnation_line(mapped_feature, Y_mp=Ymp, Z_mp=Zmp, sigma=0)
    ys += [y]
    zs += [z]
    ax[0].scatter(y,z,label=f'CLA={round(center_cla*180/np.pi,2)}°', s=2)
    clas = (temp.omni_CLA.values-center_cla+np.pi)%(2*np.pi) - np.pi + center_cla
    mean_cla = np.sum(clas*temp.weights.values)/np.sum(temp.weights.values).item()
    median_clas += [mean_cla]
ax[0].legend(bbox_to_anchor=(1.3, 1))

std_cla = np.pi/4
center_cla = 3*np.pi/4
std_tilt = 6*np.pi/180
tilts = np.linspace(-32,32,13)*np.pi/180
# Example data
#plot_CLA_sector(13, 13, 2.5, center_cla-std_cla, center_cla+std_cla, ax[1])
ys2, zs2 = [],[]
median_tilts = []
for i, center_tilt in enumerate(tilts):
    df2 = get_weights_gaussian(df, [('omni_CLA',center_cla,std_cla),('tilt',center_tilt,std_tilt)])
    #df2 = get_weights_gaussian(df, [('tilt',center_tilt,std_tilt)])
    df2['weights']=df2.weights.values/np.nansum(df2.weights.values)
    temp = df2[df2.weights.values>0]
    #mapped_feature, valid = knn_weighted_feature_map(temp,'dVz','weights',np.array([Xmp.flatten(), Ymp.flatten(), Zmp.flatten()]).T, k_neighbors=1000,max_distance_barycenter=1.5,median_distance=3)
    mapped_feature, valid = knn_weighted_feature_map(temp,'dVz','weights',np.array([Xmp.flatten(), Ymp.flatten(), Zmp.flatten()]).T, 
                                                     k_neighbors=2000,max_distance_barycenter=1.5,median_distance=3)
    mapped_feature[valid==0]=np.nan
    mapped_feature[Xmp.flatten()<0.01] = np.nan
    #mapped_feature = gaussian_filter_nan_datas((mapped_feature).reshape(Xmp.shape),9)
    mapped_feature = (mapped_feature).reshape(Xmp.shape)
    y, z = find_stagnation_line(mapped_feature, Y_mp=Ymp, Z_mp=Zmp, sigma=7)
    ys2 += [y]
    zs2 += [z]
    ax[1].scatter(y,z,label=f'Tilt={round(center_tilt*180/np.pi,2)}°', s=2)   
    median_tilts += [(temp.tilt.values*temp.weights.values).sum()/temp.weights.values.sum()]
    
ax[1].legend(bbox_to_anchor=(1.3, 1))


msh = Magnetosheath(magnetopause='mp_shue1998', bow_shock='bs_jelinek2012')
for a in ax:
    a.set_xlim(-17,17)
    a.set_ylim(-17,17)
    a.set_aspect('equal')
    a.axvline(0,linestyle='--',color='k')
    a.axhline(0,linestyle='--',color='k')
    _, _ = planet_env.layout_earth_env(msh, figure=fig, axes=np.array([a]),x_slice=0, x_lim=(-16,16),y_lim=(-16,16),z_lim=(-16,16))
       
#fig.savefig('/home/ghisalberti/Figures/Xlines.png')

In [ ]:
# [from Maps/FinalMapsPaper.ipynb, cell 288]
for liste in [ys,zs,ys2,zs2]:
    for i,val in enumerate(liste):
        liste[i]=np.array(val)

In [ ]:
# [from Maps/FinalMapsPaper.ipynb, cell 286]
df = prepare_data(pd.read_pickle(f'{OUT}/BL_data_for_maps_without_cusps_withoutNp<3_all_sat_1min_2025-05-06.pkl'))
df = df[df.uniform_depthonly_spectro.values>0.83]
Xmp, Ymp, Zmp = make_mp_grid(N_grid=100,coord='cartesian')

std_cla = np.pi/6
clas = np.linspace(np.pi/2+std_cla,np.pi,3)
min_clas = (clas-np.pi/6)[::-1]
max_clas = (clas+np.pi/6)[::-1]
nb_sectors = len(min_clas)

# Example data
ncols = nb_sectors+1
fig = plt.figure(figsize=(ncols*2.2,2.5))
gs = fig.add_gridspec(1, ncols+1, width_ratios=[1]*nb_sectors+[0.03,0.05], wspace=0.05)

# Create subplots
ax = []
for i in range(nb_sectors):
    a = fig.add_subplot(gs[0, i])
    ax.append(a)
ax=np.array(ax)

letters = ['a)','b)','c)']
for i, (min_cla, max_cla) in enumerate(zip(min_clas,max_clas)):
    std_cla = (max_cla-min_cla)/2
    center_cla = (max_cla+min_cla)/2

    df2 = get_weights_gaussian(df, [('omni_CLA',center_cla,std_cla)])
    df2['weights']=df2.weights.values/np.nansum(df2.weights.values)
    temp = df2[df2.weights.values>0]
    mapped_feature, valid = knn_weighted_feature_map(temp,'dVz','weights',np.array([Xmp.flatten(), Ymp.flatten(), Zmp.flatten()]).T, k_neighbors=2000,max_distance_barycenter=1.5,median_distance=3)
    mapped_feature[valid==0]=np.nan
    mapped_feature[Xmp.flatten()<0.01] = np.nan
    mapped_feature = gaussian_filter_nan_datas((mapped_feature).reshape(Xmp.shape),3)
    im = ax[i].pcolormesh(Ymp,Zmp,mapped_feature, cmap='seismic',vmin=-np.nanmax(abs(mapped_feature)),vmax=np.nanmax(abs(mapped_feature)))
    clas = (temp.omni_CLA.values-center_cla+np.pi)%(2*np.pi) - np.pi + center_cla
    ax[i].set_title(r'$\theta_{cl} \in$'+ f' [{round(np.nanmin(clas)*180/np.pi,2)}°;{round(np.nanmax(clas)*180/np.pi,2)}°]')
    plot_CLA_sector(13, 13, 2.5, min_cla, max_cla, ax[i])
    ax[i].text(-15,12,letters[i],fontsize='xx-large')

# Add colorbar to the last (empty) grid cell
cbar_ax = fig.add_subplot(gs[0, -1])
cbar=fig.colorbar(im, cax=cbar_ax)
#cbar.ax.tick_params(labelsize=8)
#cbar.ax.tick_params(pad=2)
cbar.set_label(r'$\Delta V_z$ (km/s)')

msh = Magnetosheath(magnetopause='mp_shue1998', bow_shock='bs_jelinek2012')
for a in ax:
    a.set_xlim(-17,17)
    a.set_ylim(-17,17)
    a.set_aspect('equal')
    a.axvline(0,linestyle='--',color='k')
    a.axhline(0,linestyle='--',color='k')
    _, _ = planet_env.layout_earth_env(msh, figure=fig, axes=np.array([a]),x_slice=0, x_lim=(-16,16),y_lim=(-16,16),z_lim=(-16,16))

for a in ax.flatten()[1:]:
    a.set_yticks([])  # Removes y-axis ticks
    a.set_ylabel('')   # Removes y-axis label

fig.savefig(f'{OUT}/figures/maps_clas_zoom1.png',bbox_inches='tight')

fig, ax = plt.subplots(figsize=(4,4))
std_cla = np.pi/6
clas = np.linspace(-np.pi,-np.pi/2-std_cla,7)
min_clas = clas-np.pi/6
max_clas = clas+np.pi/6
for i, (min_cla, max_cla) in enumerate(zip(min_clas,max_clas)):
    center_cla = (max_cla+min_cla)/2
    ax.scatter(ys[i][abs(zs[i])<9],zs[i][abs(zs[i])<9],label=r'$\theta_{cl}$'+f'={round(median_clas[i]*180/np.pi,2)}°', s=2)
ax.legend(loc='center left', bbox_to_anchor=(1, 0,0.05,1),fontsize='x-small')
ax.set_title(r'$\Delta V_z=0$'+f' lines')

msh = Magnetosheath(magnetopause='mp_shue1998', bow_shock='bs_jelinek2012')
ax.set_xlim(-17,17)
ax.set_ylim(-17,17)
ax.set_aspect('equal')
ax.axvline(0,linestyle='--',color='k')
ax.axhline(0,linestyle='--',color='k')
_, _ = planet_env.layout_earth_env(msh, figure=fig, axes=np.array([ax]),x_slice=0, x_lim=(-16,16),y_lim=(-16,16),z_lim=(-16,16))

fig.savefig(f'{OUT}/figures/maps_clas_zoom2.png',bbox_inches='tight')

In [ ]:
# [from Maps/FinalMapsPaper.ipynb, cell 292]
df = prepare_data(pd.read_pickle(f'{OUT}/BL_data_for_maps_without_cusps_withoutNp<3_all_sat_1min_2025-05-06.pkl'))
#df = df[df.uniform_depthonly_spectro.values>0.83]

std_cla = np.pi/4
center_cla = 3*np.pi/4

std_tilt = 6*np.pi/180
tilts = np.linspace(-32,0,3)*np.pi/180
min_tilts = tilts-std_tilt
max_tilts = tilts+std_tilt
nb_sectors = len(tilts)

# Example data
ncols = nb_sectors+1
fig = plt.figure(figsize=(ncols*2.2,2.5))
gs = fig.add_gridspec(1, ncols+1, width_ratios=[1]*nb_sectors+[0.03,0.05], wspace=0.01)

# Create subplots
ax = []
for i in range(nb_sectors):
    a = fig.add_subplot(gs[0, i])
    ax.append(a)
ax=np.array(ax)

letters = ['a)','b)','c)']
for i, (min_tilt, max_tilt) in enumerate(zip(min_tilts,max_tilts)):

    df2 = get_weights_gaussian(df, [('omni_CLA',center_cla,std_cla),('tilt',(max_tilt+min_tilt)/2,(max_tilt-min_tilt)/2)])
    df2['weights']=df2.weights.values/np.nansum(df2.weights.values)
    temp = df2[df2.weights.values>0]
    mapped_feature, valid = knn_weighted_feature_map(temp,'dVz','weights',np.array([Xmp.flatten(), Ymp.flatten(), Zmp.flatten()]).T, k_neighbors=1000,max_distance_barycenter=1.5,median_distance=3)
    mapped_feature[valid==0]=np.nan
    mapped_feature[Xmp.flatten()<0.01] = np.nan
    mapped_feature = gaussian_filter_nan_datas((mapped_feature).reshape(Xmp.shape),3)
    im = ax[i].pcolormesh(Ymp,Zmp,mapped_feature, cmap='seismic',vmin=-np.nanmax(abs(mapped_feature)),vmax=np.nanmax(abs(mapped_feature)))
    ax[i].set_title(r'$\Psi \in$'+ f' [{round(np.nanmin(temp.tilt.values)*180/np.pi,2)}°;{round(np.nanmax(temp.tilt.values)*180/np.pi,2)}°]')
    plot_CLA_sector(13, 13, 2.5, center_cla-std_cla, center_cla+std_cla, ax[i])
    ax[i].text(-15,12,letters[i],fontsize='xx-large')

# Add colorbar to the last (empty) grid cell
cbar_ax = fig.add_subplot(gs[0, -1])
cbar=fig.colorbar(im, cax=cbar_ax)
cbar.set_label(r'$\Delta V_z$ (km/s)')

msh = Magnetosheath(magnetopause='mp_shue1998', bow_shock='bs_jelinek2012')
for a in ax:
    a.set_xlim(-17,17)
    a.set_ylim(-17,17)
    a.set_aspect('equal')
    a.axvline(0,linestyle='--',color='k')
    a.axhline(0,linestyle='--',color='k')
    _, _ = planet_env.layout_earth_env(msh, figure=fig, axes=np.array([a]),x_slice=0, x_lim=(-16,16),y_lim=(-16,16),z_lim=(-16,16))

for a in ax.flatten()[1:]:
    a.set_yticks([])  # Removes y-axis ticks
    a.set_ylabel('')   # Removes y-axis label

fig.savefig(f'{OUT}/figures/maps_tilts_zoom1.png',bbox_inches='tight')


fig, ax = plt.subplots(figsize=(4,4))
std_tilt = 6*np.pi/180
tilts = np.linspace(-32,0,7)*np.pi/180
min_tilts = tilts-std_tilt
max_tilts = tilts+std_tilt
for i, (min_tilt, max_tilt) in enumerate(zip(min_tilts,max_tilts)):
    center_tilt = (max_tilt+min_tilt)/2
    ax.scatter(ys2[i][abs(zs2[i])<9],zs2[i][abs(zs2[i])<9],label=r'$\Psi$'+f'={round(median_tilts[i]*180/np.pi,2)}°', s=2)
ax.legend(loc='center left', bbox_to_anchor=(1, 0,0.05,1),fontsize='x-small')
ax.set_title(r'$\Delta V_z=0$'+f' lines')
msh = Magnetosheath(magnetopause='mp_shue1998', bow_shock='bs_jelinek2012')
ax.set_xlim(-17,17)
ax.set_ylim(-17,17)
ax.set_aspect('equal')
ax.axvline(0,linestyle='--',color='k')
ax.axhline(0,linestyle='--',color='k')
_, _ = planet_env.layout_earth_env(msh, figure=fig, axes=np.array([ax]),x_slice=0, x_lim=(-16,16),y_lim=(-16,16),z_lim=(-16,16))

fig.savefig(f'{OUT}/figures/maps_tilts_zoom2.png',bbox_inches='tight')


## Paper figure: comparison of the BL X-line with the models

In [ ]:
# [from Maps/FinalMapsPaper.ipynb, cell 630]
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import GridSpec
import matplotlib.patheffects as PathEffects
from blxline.maps import gaussian_filter_nan_1d

run = 0

df = prepare_data(pd.read_pickle(f'{OUT}/BL_data_for_maps_without_cusps_withoutNp<3_all_sat_1min_2025-05-06.pkl'))
df = df[df.uniform_depthonly_spectro.values>0.83]
Xmp, Ymp, Zmp = make_mp_grid(N_grid=100,coord='cartesian')

std_cla = 30/180*np.pi
clas = np.array([120,150,180])[::-1]
min_clas = (clas-std_cla)
max_clas = (clas+std_cla)
nb_sectors_cla = len(min_clas)

std_tilt = 7*np.pi/180
tilts = np.array([-30,-15,0])[::-1]
min_tilts = tilts-std_tilt
max_tilts = tilts+std_tilt
nb_sectors_tilt = len(min_tilts)

# Example data
ncols = nb_sectors_cla
nrows = nb_sectors_tilt
fig = plt.figure(figsize=(ncols*3+1.5,nrows*3))
gs = fig.add_gridspec(nrows, ncols+2, width_ratios=[1]*ncols+[0.05,0.05], hspace=0.05, wspace=0.05) 

# Create subplots
ax = []
for j in range(nb_sectors_tilt):
    ax_row = []
    for i in range(nb_sectors_cla):
        a = fig.add_subplot(gs[j,i])
        ax_row.append(a)
    ax.append(ax_row)
ax=np.array(ax)

letters = ['a)','b)','c)','d)',
           'e)','f)','g)','h)',
           'i)','j)','k)','l)',
           'm)','n)','o)','p)',
           'q)','r)','s)','t)',
          'u)','v)','w)','x)']

if run==0:
    ys, zs, mapped_features = [[] for _ in range(nb_sectors_cla)],[[] for _ in range(nb_sectors_cla)],[[] for _ in range(nb_sectors_cla)]
    vmax = np.zeros(nb_sectors_cla)
else:
    vmax = np.zeros(nb_sectors_cla)
    for i,maps in enumerate(mapped_features):
        vmax_i = 0
        for mappy in maps:
            vmax_i = max(np.nanmax(abs(mappy)),vmax_i)
        vmax[i]= vmax_i

count = 0
for j, tilt in enumerate(tilts):
    for i, cla in enumerate(clas):
        coord = pd.read_pickle(f'{MODEL_LINES}/DXL_cla_{str(cla)}_coa_90_tilt_{str(tilt)}.pkl')
        y,z = coord['y'], coord['z']
        coord = pd.read_pickle(f'{MODEL_LINES}/MRL_cla{str(cla)}_tilt{str(tilt)}.pkl')
        y3, z3 = coord['y'], coord['z']
        if cla!=180:
            coord = pd.read_pickle(f'{MODEL_LINES}/MSL_cla_{str(cla)}_coa_90_tilt_{str(tilt)}.pkl')
            y2,z2 = coord['y'], coord['z']

        if run==0:
            df2 = get_weights_gaussian(df, [('omni_CLA',cla*np.pi/180,std_cla),('tilt',tilt*np.pi/180,std_tilt)])
            df2['weights']=df2.weights.values/np.nansum(df2.weights.values)
            temp = df2[df2.weights.values>0]
            mapped_feature, valid = knn_weighted_feature_map(temp,'dVz','weights',
                                                             np.array([Xmp.flatten(), Ymp.flatten(), Zmp.flatten()]).T, 
                                                             k_neighbors=1000,max_distance_barycenter=1.5,median_distance=3)
            mapped_feature[valid==0]=np.nan
            mapped_feature[Xmp.flatten()<0.01] = np.nan
            mapped_feature_smoothed = gaussian_filter_nan_datas((mapped_feature).reshape(Xmp.shape),3)
            mapped_features[i] += [mapped_feature]
            vmax[i] = np.nanmax(abs(mapped_feature_smoothed))

            y_measure,z_measure = find_stagnation_line(mapped_feature.reshape(Xmp.shape),Y_mp=Ymp, Z_mp=Zmp, sigma=7)
            ys[i] += [y_measure]
            zs[i] += [z_measure]

        else:
            mapped_feature = mapped_features[i][j]
            mapped_feature_smoothed = gaussian_filter_nan_datas((mapped_feature).reshape(Xmp.shape),3)
            y_measure,z_measure = ys[i][j], zs[i][j]

        y_measure,z_measure = keep_main_part(y_measure,z_measure)
        z_measure = gaussian_filter_nan_1d(z_measure,1)
        im = ax[j,i].pcolormesh(Ymp,Zmp,mapped_feature_smoothed, cmap='seismic',vmin=-vmax[i],vmax=vmax[i])
        plot_CLA_sector(13, 13, 2.5, cla*np.pi/180-std_cla, cla*np.pi/180+std_cla, ax[j,i])
        ax[j,i].text(-15,12,letters[count],fontsize='xx-large')
        line,=ax[j,i].plot(y,z, linewidth=3, color='w',label='DXL')
        line.set_path_effects([
        PathEffects.Stroke(linewidth=5, foreground='black'), # The border
        PathEffects.Normal()])
        
        ax[j,i].plot(y3,z3, linewidth=3, color='orange',label='MRL')
        
        if cla!=180:
            ax[j,i].plot(y2,z2, linewidth=4, color='grey',label='MSL')

        if cla==180 and tilt==-10:
            nb = 30
        elif cla==155 and tilt==-10:
            nb = 30
        elif cla==135 and tilt==-22:
            nb = 10
        elif cla==120 and tilt==-30:
            nb = 10
        elif cla==120 and tilt==-22:
            nb = 20
        else:
            nb=1
        ax[j,i].plot(y_measure[nb:],z_measure[nb:], label=r'$\Delta V_z = 0$ BL', color='purple', linewidth=4)
        count += 1
                
    # Add colorbar to the last (empty) grid cell
    cbar_ax = fig.add_subplot(gs[j, -1])
    cbar=fig.colorbar(im, cax=cbar_ax)
    #cbar.ax.tick_params(labelsize=8)
    #cbar.ax.tick_params(pad=2)
    cbar.set_label("km/s",labelpad=-20)
    
        
for i in range(len(min_clas)):
    #ax[i,0].text(1/(len(min_clas)+2)*(i+1),1-0.05,f'CLA '+r'$\in$'+ f' [{round(min_clas[i]*180/np.pi,2)}°;{round(max_clas[i]*180/np.pi,2)}°]', transform=plt.gcf().transFigure)
    ax[0,i].annotate(f'CLA '+r'$\in$'+ f' [{round(clas[i] - std_cla*180/np.pi,2)}°;{round(clas[i]+std_cla*180/np.pi,2)}°]',xy=(0.5, 1), xytext=(0, 20),
                        xycoords='axes fraction', textcoords='offset points',
                        size='large', ha='center', va='baseline', weight='bold')
for j in range(len(min_tilts)):
    #ax[0,j].text(0.05,1-1/(len(min_tilts)+2)*(j+1),f'Tilt '+r'$\in$'+ f' [{round(min_tilts[j]*180/np.pi,2)}°;{round(max_tilts[j]*180/np.pi,2)}°]', rotation = 90, transform=plt.gcf().transFigure)
    ax[j,0].annotate(f'Tilt '+r'$\in$'+ f'[{round(tilts[j]-std_tilt*180/np.pi,2)}°;{round(tilts[j]+std_tilt*180/np.pi,2)}°]', xy=(-0.1, 0.5), xytext=(-40, 0),
                        xycoords='axes fraction', textcoords='offset points',
                        size='large', ha='right', va='center', weight='bold', rotation=90)

msh = Magnetosheath(magnetopause='mp_shue1998', bow_shock='bs_jelinek2012')
for a in ax.flatten():
    a.set_xlim(-17,17)
    a.set_ylim(-17,17)
    a.set_aspect('equal')
    a.axvline(0,linestyle='--',color='k')
    a.axhline(0,linestyle='--',color='k')
    _, _ = planet_env.layout_earth_env(msh, figure=fig, axes=np.array([a]),x_slice=0, x_lim=(-16,16),y_lim=(-16,16),z_lim=(-16,16))
ax[-1,-1].legend(loc='lower right')

for a in ax[:,1:].flatten():
    a.set_yticks([])  # Removes y-axis ticks
    a.set_ylabel('')   # Removes y-axis label
for a in ax[:-1,:].flatten():
    a.set_xticks([])  # Removes x-axis ticks
    a.set_xlabel('')   # Removes x-axis label

#fig.tight_layout()

fig.savefig(f'{OUT}/figures/comparison_models_jgr3.png',bbox_inches='tight')

In [ ]:
# [from Maps/FinalMapsPaper.ipynb, cell 630]
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import GridSpec
import matplotlib.patheffects as PathEffects
from blxline.maps import gaussian_filter_nan_1d

run = 1

df = prepare_data(pd.read_pickle(f'{OUT}/BL_data_for_maps_without_cusps_withoutNp<3_all_sat_1min_2025-05-06.pkl'))
df = df[df.uniform_depthonly_spectro.values>0.83]
Xmp, Ymp, Zmp = make_mp_grid(N_grid=100,coord='cartesian')

std_cla = 30/180*np.pi
clas = np.array([120,150,180])[::-1]
min_clas = (clas-std_cla)
max_clas = (clas+std_cla)
nb_sectors_cla = len(min_clas)

std_tilt = 7*np.pi/180
tilts = np.array([-30,-15,0])[::-1]
min_tilts = tilts-std_tilt
max_tilts = tilts+std_tilt
nb_sectors_tilt = len(min_tilts)

# Example data
ncols = nb_sectors_cla
nrows = nb_sectors_tilt
fig = plt.figure(figsize=(ncols*3+1.5,nrows*3))
gs = fig.add_gridspec(nrows, ncols+2, width_ratios=[1]*ncols+[0.05,0.05], hspace=0.05, wspace=0.05) 

# Create subplots
ax = []
for j in range(nb_sectors_tilt):
    ax_row = []
    for i in range(nb_sectors_cla):
        a = fig.add_subplot(gs[j,i])
        ax_row.append(a)
    ax.append(ax_row)
ax=np.array(ax)

letters = ['a)','b)','c)','d)',
           'e)','f)','g)','h)',
           'i)','j)','k)','l)',
           'm)','n)','o)','p)',
           'q)','r)','s)','t)',
          'u)','v)','w)','x)']

if run==0:
    ys, zs, mapped_features = [[] for _ in range(nb_sectors_cla)],[[] for _ in range(nb_sectors_cla)],[[] for _ in range(nb_sectors_cla)]
    vmax = np.zeros(nb_sectors_cla)
else:
    vmax = np.zeros(nb_sectors_cla)
    for i,maps in enumerate(mapped_features):
        vmax_i = 0
        for mappy in maps:
            vmax_i = max(np.nanmax(abs(mappy)),vmax_i)
        vmax[i]= vmax_i

count = 0
for j, tilt in enumerate(tilts):
    for i, cla in enumerate(clas):
        coord = pd.read_pickle(f'{MODEL_LINES}/DXL_cla_{str(cla)}_coa_90_tilt_{str(tilt)}.pkl')
        y,z = coord['y'], coord['z']
        coord = pd.read_pickle(f'{MODEL_LINES}/MRL_cla{str(cla)}_tilt{str(tilt)}.pkl')
        y3, z3 = coord['y'], coord['z']
        if cla!=180:
            coord = pd.read_pickle(f'{MODEL_LINES}/MSL_cla_{str(cla)}_coa_90_tilt_{str(tilt)}.pkl')
            y2,z2 = coord['y'], coord['z']

        if run==0:
            df2 = get_weights_gaussian(df, [('omni_CLA',cla*np.pi/180,std_cla),('tilt',tilt*np.pi/180,std_tilt)])
            df2['weights']=df2.weights.values/np.nansum(df2.weights.values)
            temp = df2[df2.weights.values>0]
            mapped_feature, valid = knn_weighted_feature_map(temp,'dVz','weights',
                                                             np.array([Xmp.flatten(), Ymp.flatten(), Zmp.flatten()]).T, 
                                                             k_neighbors=1000,max_distance_barycenter=1.5,median_distance=3)
            mapped_feature[valid==0]=np.nan
            mapped_feature[Xmp.flatten()<0.01] = np.nan
            mapped_feature_smoothed = gaussian_filter_nan_datas((mapped_feature).reshape(Xmp.shape),3)
            mapped_features[i] += [mapped_feature]
            vmax[i] = np.nanmax(abs(mapped_feature_smoothed))

            y_measure,z_measure = find_stagnation_line(mapped_feature.reshape(Xmp.shape),Y_mp=Ymp, Z_mp=Zmp, sigma=7)
            ys[i] += [y_measure]
            zs[i] += [z_measure]

        else:
            mapped_feature = mapped_features[i][j]
            mapped_feature_smoothed = gaussian_filter_nan_datas((mapped_feature).reshape(Xmp.shape),3)
            y_measure,z_measure = ys[i][j], zs[i][j]

        y_measure,z_measure = keep_main_part(y_measure,z_measure)
        z_measure = gaussian_filter_nan_1d(z_measure,1)
        im = ax[j,i].pcolormesh(Ymp,Zmp,mapped_feature_smoothed, cmap='seismic',vmin=-vmax[i],vmax=vmax[i])
        plot_CLA_sector(13, 13, 2.5, cla*np.pi/180-std_cla, cla*np.pi/180+std_cla, ax[j,i])
        ax[j,i].text(-15,12,letters[count],fontsize='xx-large')
        line,=ax[j,i].plot(y,z, linewidth=3, color='w',label='DXL')
        line.set_path_effects([
        PathEffects.Stroke(linewidth=5, foreground='black'), # The border
        PathEffects.Normal()])
        
        ax[j,i].plot(y3,z3, linewidth=3, color='orange',label='MRL')
        
        if cla!=180:
            ax[j,i].plot(y2,z2, linewidth=4, color='grey',label='MSL')

        if cla==180 and tilt==-10:
            nb = 30
        elif cla==155 and tilt==-10:
            nb = 30
        elif cla==135 and tilt==-22:
            nb = 10
        elif cla==120 and tilt==-30:
            nb = 10
        elif cla==120 and tilt==-22:
            nb = 20
        else:
            nb=1
        ax[j,i].plot(y_measure[nb:],z_measure[nb:], label=r'$\Delta V_z = 0$ BL', color='purple', linewidth=4)
        count += 1
                
    # Add colorbar to the last (empty) grid cell
    cbar_ax = fig.add_subplot(gs[j, -1])
    cbar=fig.colorbar(im, cax=cbar_ax)
    #cbar.ax.tick_params(labelsize=8)
    #cbar.ax.tick_params(pad=2)
    cbar.set_label("km/s",labelpad=-20)
    
        
for i in range(len(min_clas)):
    #ax[i,0].text(1/(len(min_clas)+2)*(i+1),1-0.05,f'CLA '+r'$\in$'+ f' [{round(min_clas[i]*180/np.pi,2)}°;{round(max_clas[i]*180/np.pi,2)}°]', transform=plt.gcf().transFigure)
    ax[0,i].annotate(f'CLA '+r'$\in$'+ f' [{round(clas[i] - std_cla*180/np.pi,2)}°;{round(clas[i]+std_cla*180/np.pi,2)}°]',xy=(0.5, 1), xytext=(0, 20),
                        xycoords='axes fraction', textcoords='offset points',
                        size='large', ha='center', va='baseline', weight='bold')
for j in range(len(min_tilts)):
    #ax[0,j].text(0.05,1-1/(len(min_tilts)+2)*(j+1),f'Tilt '+r'$\in$'+ f' [{round(min_tilts[j]*180/np.pi,2)}°;{round(max_tilts[j]*180/np.pi,2)}°]', rotation = 90, transform=plt.gcf().transFigure)
    ax[j,0].annotate(f'Tilt '+r'$\in$'+ f'[{round(tilts[j]-std_tilt*180/np.pi,2)}°;{round(tilts[j]+std_tilt*180/np.pi,2)}°]', xy=(-0.1, 0.5), xytext=(-40, 0),
                        xycoords='axes fraction', textcoords='offset points',
                        size='large', ha='right', va='center', weight='bold', rotation=90)

msh = Magnetosheath(magnetopause='mp_shue1998', bow_shock='bs_jelinek2012')
for a in ax.flatten():
    a.set_xlim(-17,17)
    a.set_ylim(-17,17)
    a.set_aspect('equal')
    a.axvline(0,linestyle='--',color='k')
    a.axhline(0,linestyle='--',color='k')
    _, _ = planet_env.layout_earth_env(msh, figure=fig, axes=np.array([a]),x_slice=0, x_lim=(-16,16),y_lim=(-16,16),z_lim=(-16,16))
ax[-1,-1].legend(loc='lower right')

for a in ax[:,1:].flatten():
    a.set_yticks([])  # Removes y-axis ticks
    a.set_ylabel('')   # Removes y-axis label
for a in ax[:-1,:].flatten():
    a.set_xticks([])  # Removes x-axis ticks
    a.set_xlabel('')   # Removes x-axis label

#fig.tight_layout()

fig.savefig(f'{OUT}/figures/comparison_models_jgr3.png',bbox_inches='tight')